# Image Warping and Mosaicing
Rishbha Jain · CS280A · Project 4, Part A

## A.1 — Photos


In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import display
from panorama_extensions import (
    load_photos, align_photos, make_panorama, save_panorama, EXAMPLE_GROUPS,
)

photos, original_sizes = load_photos("images", max_side=1200, mode="RGB")
output_folder = Path("outputs/panoramas")
output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 14))
for ax, (name, photo) in zip(axes.flat, photos.items()):
    ax.imshow(photo)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
fig.savefig(output_folder / "source_photo_gallery.png", dpi=150)
plt.show()

## A.2 — Homography


In [ ]:
def homography_system(im1_pts, im2_pts):
    source = np.asarray(im1_pts, dtype=float)
    target = np.asarray(im2_pts, dtype=float)
    if source.shape != target.shape or source.ndim != 2 or source.shape[1] != 2:
        raise ValueError("The two point arrays must have the same shape (n, 2).")
    if len(source) < 4 or not np.isfinite(source).all() or not np.isfinite(target).all():
        raise ValueError("At least four finite point pairs are required.")
    A = np.zeros((2 * len(source), 8))
    for i, ((x, y), (u, v)) in enumerate(zip(source, target)):
        A[2*i] = [x, y, 1, 0, 0, 0, -u*x, -u*y]
        A[2*i+1] = [0, 0, 0, x, y, 1, -v*x, -v*y]
    return A, target.reshape(-1)

def computeH(im1_pts, im2_pts):
    A, b = homography_system(im1_pts, im2_pts)
    h, _, rank, _ = np.linalg.lstsq(A, b, rcond=None)
    if rank < 8:
        raise ValueError("The point pairs do not determine a homography.")
    return np.append(h, 1).reshape(3, 3)

In [ ]:
data = json.loads(Path("img1_img2.json").read_text())
im1_pts = np.array(data["im1Points"], dtype=float)
im2_pts = np.array(data["im2Points"], dtype=float)
A, b = homography_system(im1_pts, im2_pts)
H = computeH(im1_pts, im2_pts)
with np.printoptions(precision=6, suppress=True, linewidth=130):
    print("A =", A, "\nb =", b, "\nH =", H, sep="\n")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 12), gridspec_kw={"height_ratios": [3, 1, 1]})
colors = plt.get_cmap("tab10")(np.arange(9))
names = ["IMG_6205.HEIC", "IMG_6206.HEIC"]
for col, (name, points) in enumerate(zip(names, [im1_pts, im2_pts])):
    width, height = original_sizes[name]
    for row in range(3):
        ax = axes[row, col]
        ax.imshow(photos[name], extent=(-.5, width-.5, height-.5, -.5))
        ax.scatter(points[:, 0], points[:, 1], c=colors, s=25, edgecolors="black", linewidths=.5)
        ids = range(9) if row == 0 else (range(1, 5) if row == 1 else range(5, 9))
        for k in ids:
            ax.annotate(str(k+1), points[k], xytext=(5, 4), textcoords="offset points",
                        color="white", fontsize=9, bbox=dict(facecolor="black", alpha=.7, pad=1))
        ax.axis("off")
    axes[0, col].set_title(name)
    cluster = points[1:5]
    axes[1, col].set_xlim(cluster[:, 0].min()-70, cluster[:, 0].max()+70)
    axes[1, col].set_ylim(cluster[:, 1].max()+40, cluster[:, 1].min()-40)
    cluster = points[5:9]
    axes[2, col].set_xlim(cluster[:, 0].min()-70, cluster[:, 0].max()+70)
    axes[2, col].set_ylim(cluster[:, 1].max()+40, cluster[:, 1].min()-40)
plt.tight_layout()
fig.savefig(output_folder / "manual_correspondences.png", dpi=150)
plt.show()

## A.3 — Rectification


In [ ]:
from part_a import warpImageNearestNeighbor, warpImageBilinear
from build_rectification_examples import main as build_rectifications

build_rectifications()
rectification = json.loads(Path("outputs/rectification/results.json").read_text())
for case in rectification["cases"]:
    display(Image.open(f'outputs/rectification/{case["name"]}_comparison.png'))
    display(Image.open(f'outputs/rectification/{case["name"]}_detail.png'))
    print(case["title"], {k: round(v, 2) for k, v in case["median_ms"].items()}, "ms")

## A.4 — Mosaics
Additional correspondences use SIFT and a custom RANSAC filter, with 64 saved matches per pair. Coordinates are normalized before least squares. The canvas covers the transformed image boundaries; bilinear samples are blended with weights that increase away from source edges.

In [ ]:
transforms, diagnostics = align_photos(photos, original_sizes)
for row in diagnostics:
    print(f'{row["source"]} → {row["target"]}: median fit error {row["median_error_px"]:.3f} px')

In [ ]:
from panorama_extensions import pair_points

point_data = json.loads(Path("panorama_correspondences.json").read_text())
examples = [
    ("IMG_6205.HEIC", "IMG_6206.HEIC"),
    ("IMG_6208.HEIC", "IMG_6209.HEIC"),
    ("IMG_6211.HEIC", "IMG_6212.HEIC"),
]
fig, axes = plt.subplots(3, 2, figsize=(12, 15))
for row_index, (source_name, target_name) in enumerate(examples):
    pair = next(p for p in point_data["pairs"]
                if p["source"] == source_name and p["target"] == target_name)
    source_points, target_points = pair_points(pair, photos, original_sizes)
    selection = np.linspace(0, len(source_points)-1, 16).astype(int)
    colors = plt.get_cmap("tab20")(np.linspace(0, 1, len(selection)))
    for ax, name, points in zip(axes[row_index],
                               [source_name, target_name],
                               [source_points, target_points]):
        shown = points[selection]
        ax.imshow(photos[name])
        ax.scatter(shown[:, 0], shown[:, 1], c=colors,
                   s=35, edgecolors="black", linewidths=0.5)
        for number, (x, y) in enumerate(shown, 1):
            ax.annotate(str(number), (x, y), xytext=(5, 4),
                        textcoords="offset points", color="white", fontsize=9,
                        bbox=dict(facecolor="black", alpha=0.65, pad=1))
        ax.set_title(name)
        ax.axis("off")
plt.tight_layout()
fig.savefig(output_folder / "additional_correspondences.png", dpi=160)
plt.show()

In [ ]:
print("Homographies: working-image coordinates -> IMG_6209 reference coordinates")
with np.printoptions(precision=6, suppress=True):
    for name, H_to_reference in transforms.items():
        print(f"\n{name}:")
        print(H_to_reference)

In [ ]:
mosaics = {}
for label, (names, reference) in EXAMPLE_GROUPS.items():
    result, mask, info = make_panorama(
        photos, transforms, names=names, reference=reference,
        projection="planar"
    )
    mosaics[label] = result
    save_panorama(result, mask, output_folder / f"{label}.png")

    fig, axes = plt.subplots(1, 4, figsize=(18, 7))
    for ax, name in zip(axes[:3], names):
        ax.imshow(photos[name])
        ax.set_title(name)
        ax.axis("off")
    axes[3].imshow(result)
    axes[3].set_title("Feather-blended mosaic")
    axes[3].axis("off")
    fig.suptitle(label.replace("_", " ").title())
    plt.tight_layout()
    fig.savefig(output_folder / f"{label}_notebook_comparison.png", dpi=160)
    plt.show()
    print(f'Canvas: {info["width"]} x {info["height"]} pixels')

## A.5 — Cylindrical projection
For cylinder coordinates $(u,v)$, use $\theta=u/f$ and ray $r=[\sin\theta,v/f,\cos\theta]^T$. Sample source image $i$ at $H_i^{-1}Kr$, dividing by the third coordinate. $K$ uses focal length $f$ and the reference image center. This samples each source directly. The choice $f=900$ is visual, not calibrated.

In [ ]:
reference = "IMG_6209.HEIC"

planar, planar_mask, planar_info = make_panorama(
    photos, transforms, reference=reference, projection="planar"
)

focal_length = photos[reference].shape[1]
cylindrical, cylindrical_mask, cylindrical_info = make_panorama(
    photos, transforms,
    reference=reference,
    projection="cylindrical",
    focal_length=focal_length
)

save_panorama(planar, planar_mask, output_folder / "all_images_planar.png")
save_panorama(cylindrical, cylindrical_mask,
              output_folder / "all_images_cylindrical.png")

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(planar)
axes[0].set_title("All nine images: planar projection")
axes[1].imshow(cylindrical)
axes[1].set_title(f"All nine images: cylindrical projection, f={focal_length}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
fig.savefig(output_folder / "planar_vs_cylindrical.png", dpi=180)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
for ax, factor in zip(axes, [2/3, 1, 4/3]):
    f = focal_length * factor
    result, mask, info = make_panorama(
        photos, transforms, reference=reference,
        projection="cylindrical", focal_length=f
    )
    save_panorama(result, mask,
                  output_folder / f"all_images_cylindrical_f{f:.0f}.png")
    ax.imshow(result)
    ax.set_title(f"f = {f:.0f} working-image pixels")
    ax.axis("off")
plt.tight_layout()
fig.savefig(output_folder / "cylindrical_focal_comparison.png", dpi=160)
plt.show()